# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines and verifies the data contract for the **Refresh / Content Opportunity Scoring** lane. It uses pseudonymized warehouse data only and keeps the final month sealed for later testing.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-data-contracts` plus `flyrank/flyrank-data`.

## 1. Contract in plain words — five answers

1. **What does one row mean?** One modeling row is one pseudonymized content page for one pseudonymized client, identified by (`client_hash_id`, `content_hash_id`). The source's daily rows are aggregated to that page-level decision grain.
2. **Which table will I use?** I use `fact_content_daily_performance`, specifically its March 2026 partition.
3. **Which time window will I use?** Features use March 1–15, 2026; I score after March 15; the observed label uses March 16–31, 2026.
4. **What will I predict or rank?** I rank pages for human review. `is_future_decline = 1` when average daily impressions in March 16–31 are more than 20% below the March 1–15 average. Eligible pages have at least 100 early-window impressions, `gsc_data_available IS TRUE`, and an observed later-window row.
5. **What do I deliberately exclude?** I exclude March 16–31 impressions and every other later-window value from the features because they are unavailable at the March 15 decision moment and would leak the answer.

March is a mid-panel month. June 2026 and the `_sample` table remain sealed for final testing. The score predicts a later observed decline; it does not claim that refreshing a page will cause recovery.

In [1]:
import importlib.util
import os
import subprocess
import sys

required_packages = {"duckdb": "duckdb", "pandas": "pandas", "sklearn": "scikit-learn"}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import duckdb
import pandas as pd

# A local root is supported for reproducible tests; normal Colab use reads the gated HF release.
data_root = os.environ.get("FLYRANK_DATA_ROOT")
if data_root:
    REL = data_root.rstrip("/")
else:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN; never paste it into this public notebook.")
    REL = "hf://datasets/FlyRank/internship-warehouse"

con = duckdb.connect()
if not data_root:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    del hf_token, escaped_token

MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("Contract windows configured: March 1-15 features -> March 16-31 observed label.")

Contract windows configured: March 1-15 features -> March 16-31 observed label.


## 2. Fields: feature / label / context / excluded

### Five features maximum

1. **`log_early_impressions`** — knowable at the decision moment because it uses only impressions recorded from March 1–15.
2. **`early_ctr`** — knowable because early-window clicks and impressions are complete before scoring.
3. **`early_active_days`** — knowable because it counts early-window days with at least one impression.
4. **`early_avg_position`** — knowable because it summarizes observed positions from March 1–15 only.
5. **`early_position_volatility`** — knowable because it measures early-window daily-position variation only.

### Other field roles

- **Label:** `is_future_decline`, calculated from average daily impressions in March 16–31 compared with March 1–15. It is never a feature.
- **Context:** `client_hash_id` and `content_hash_id` are used only for grouping, joins, uniqueness checks, and client-held-out splitting.
- **Availability/filter:** `gsc_data_available` is used with `IS TRUE`; it is not a model feature.
- **Excluded:** March 16–31 impressions and every later-window-derived column are excluded from features because they occur after the decision moment. `ga4_*` fields are excluded from this first frame because the lane is GSC-based and GA4 history is incomplete for many rows. Raw client names, domains, URLs, queries, and product decision flags are not present and must never be reconstructed.

In [2]:
field_contract = pd.DataFrame([
    ("log_early_impressions", "feature", "March 1-15 observations only"),
    ("early_ctr", "feature", "Early-window clicks divided by impressions"),
    ("early_active_days", "feature", "Early-window days with impressions"),
    ("early_avg_position", "feature", "Early-window impression-weighted position"),
    ("early_position_volatility", "feature", "Early-window daily-position variation"),
    ("is_future_decline", "label", "Observed from March 16-31 outcome; never a feature"),
    ("client_hash_id", "context", "Grouping and held-out split only"),
    ("content_hash_id", "context", "Grouping and uniqueness only"),
    ("gsc_data_available", "filter", "Keep rows only when IS TRUE"),
    ("late_impressions", "excluded", "Future label component; using it as a feature leaks the answer"),
], columns=["field", "role", "reason"])
field_contract

                       field  ...                                             reason
0      log_early_impressions  ...                       March 1-15 observations only
1                  early_ctr  ...         Early-window clicks divided by impressions
2          early_active_days  ...                 Early-window days with impressions
3         early_avg_position  ...          Early-window impression-weighted position
4  early_position_volatility  ...              Early-window daily-position variation
5          is_future_decline  ...  Observed from March 16-31 outcome; never a fea...
6             client_hash_id  ...                   Grouping and held-out split only
7            content_hash_id  ...                       Grouping and uniqueness only
8         gsc_data_available  ...                        Keep rows only when IS TRUE
9           late_impressions  ...  Future label component; using it as a feature ...

[10 rows x 3 columns]

## 3. Verify it with queries (grain, counts, availability, five-feature frame, leakage trap)

The first three SQL statements are the required verification queries. They test raw-table grain, the March slice's size/date span, and the number of rows surviving `gsc_data_available IS TRUE`. The fourth SQL statement builds the five-feature modeling frame; it is a feature-building query, not an additional contract claim.

In [3]:
# Verification query 1 — declared modeling grain. Zero returned rows means one row per client-page.
grain_query = f"""
WITH modeling_rows AS (
    SELECT client_hash_id, content_hash_id
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM modeling_rows
GROUP BY 1, 2
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_duplicates = con.sql(grain_query).df()
print("Verification query 1 — duplicate modeling-grain rows:", len(grain_duplicates))
print(grain_duplicates.to_string(index=False) if len(grain_duplicates) else "No duplicates returned; one row per eligible client-page.")

# Verification query 2 — March row count and observed date window.
count_window_query = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MARCH}
"""
count_window_check = con.sql(count_window_query).df()
print("\nVerification query 2 — March slice:")
print(count_window_check.to_string(index=False))

# Verification query 3 — availability. IS TRUE handles FALSE and NULL honestly.
availability_query = f"""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows_surviving,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows_surviving,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / NULLIF(COUNT(*), 0), 2) AS gsc_survival_pct
FROM {MARCH}
"""
availability_check = con.sql(availability_query).df()
print("\nVerification query 3 — availability:")
print(availability_check.to_string(index=False))

# Feature-building query — one page-level row, five early-March features, later-March observed label.
feature_query = f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS early_impressions,
        SUM(gsc_clicks) AS early_clicks,
        COUNT(*) AS early_observed_days,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS early_active_days,
        SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                            THEN gsc_impressions ELSE 0 END), 0) AS early_avg_position,
        STDDEV_SAMP(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS early_position_volatility
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS late_impressions,
        COUNT(*) AS late_observed_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
)
SELECT
    e.client_hash_id,
    e.content_hash_id,
    LN(1 + e.early_impressions) AS log_early_impressions,
    1.0 * e.early_clicks / NULLIF(e.early_impressions, 0) AS early_ctr,
    e.early_active_days,
    e.early_avg_position,
    e.early_position_volatility,
    CAST((1.0 * l.late_impressions / NULLIF(l.late_observed_days, 0))
         < 0.8 * (1.0 * e.early_impressions / NULLIF(e.early_observed_days, 0)) AS INTEGER)
      AS is_future_decline
FROM early e
JOIN late l USING (client_hash_id, content_hash_id)
"""
feature_frame = con.sql(feature_query).df()
assert not feature_frame.duplicated(["client_hash_id", "content_hash_id"]).any()
feature_columns = [
    "log_early_impressions", "early_ctr", "early_active_days",
    "early_avg_position", "early_position_volatility",
]
print(f"\nFeature frame: {len(feature_frame):,} eligible page rows; exactly {len(feature_columns)} features.")
print(feature_frame[feature_columns + ["is_future_decline"]].head().to_string(index=False))

# Deliberate leakage experiment — copy the label, observe the jump, then delete it.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_frame = feature_frame.dropna(subset=["is_future_decline"]).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(model_frame, model_frame["is_future_decline"], groups=model_frame["client_hash_id"]))
y_train = model_frame.iloc[train_idx]["is_future_decline"]
y_test = model_frame.iloc[test_idx]["is_future_decline"]

def fit_auc(columns, frame):
    pipeline = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42),
    )
    pipeline.fit(frame.iloc[train_idx][columns], y_train)
    probabilities = pipeline.predict_proba(frame.iloc[test_idx][columns])[:, 1]
    return roc_auc_score(y_test, probabilities)

honest_auc = fit_auc(feature_columns, model_frame)
leaky_frame = model_frame.copy()
leaky_frame["leaked_label_copy"] = leaky_frame["is_future_decline"]
leaky_auc = fit_auc(feature_columns + ["leaked_label_copy"], leaky_frame)
del leaky_frame["leaked_label_copy"]
assert "leaked_label_copy" not in leaky_frame.columns
print(f"\nHonest held-out-client ROC-AUC: {honest_auc:.3f}")
print(f"Deliberately leaked ROC-AUC: {leaky_auc:.3f}")
print("Leak column deleted. The honest score is the retained result.")

Verification query 1 — duplicate modeling-grain rows: 0
No duplicates returned; one row per eligible client-page.

Verification query 2 — March slice:
 row_count  clients  content_items first_date  last_date
   9841378       55         331437 2026-03-01 2026-03-31

Verification query 3 — availability:
 all_rows  gsc_rows_surviving  ga4_rows_surviving  gsc_survival_pct
  9841378             3611061              413966             36.69

Feature frame: 77,400 eligible page rows; exactly 5 features.
 log_early_impressions  early_ctr  early_active_days  early_avg_position  early_position_volatility  is_future_decline
              6.851185   0.007415                 15            2.796610                   0.617851                  1
              4.912655   0.000000                 15           10.592593                  10.872382                  0
              8.065894   0.004084                 15            2.454288                   0.285252                  0
              5.768321

## 4. Data limits

- Client histories are unbalanced. The split-March window represents only clients and pages with usable history in both halves of the month.
- `gsc_data_available` can be `TRUE`, `FALSE`, or `NULL`; zero-filled or unavailable periods are not evidence of zero demand.
- The inner early-to-late join excludes pages without an observed March 16–31 row, which can create survival bias.
- The label measures later impression decline, not the causal benefit of refreshing content. A high score means “review this page,” not “a refresh will definitely work.”
- Daily average position and its volatility can be missing or unstable for low-volume pages. The 100-impression floor reduces, but does not eliminate, that noise.
- Pseudonymized IDs permit grouped validation but cannot explain client-specific business context. No re-identification is attempted.

**Named limitation:** the eligible frame may over-represent established, consistently tracked pages, so findings should not be generalized to newly published or poorly tracked content without a separate study.

In [4]:
limitation_summary = pd.Series({
    "eligible_page_rows": len(feature_frame),
    "represented_clients": feature_frame["client_hash_id"].nunique(),
    "future_decline_rate": feature_frame["is_future_decline"].mean(),
    "missing_avg_position_pct": 100 * feature_frame["early_avg_position"].isna().mean(),
    "missing_position_volatility_pct": 100 * feature_frame["early_position_volatility"].isna().mean(),
})
limitation_summary

eligible_page_rows                 77400.000000
represented_clients                   38.000000
future_decline_rate                    0.345465
missing_avg_position_pct               0.000000
missing_position_volatility_pct        0.002584
dtype: float64

## Self-check

Before submitting, confirm each line honestly:

- [x] Five plain-language contract answers are present
- [x] Exactly three verification queries have visible outputs
- [x] Availability is checked with `IS TRUE`
- [x] The frame contains exactly five decision-time features
- [x] The deliberate leak is demonstrated and removed
- [x] One limitation is named
- [x] The notebook runs top to bottom with no errors
- [x] No client names, domains, URLs, raw queries, tokens, or private data appear
- [ ] Committed under `work/notebooks/` and the repository URL submitted on the card